# Solution C2 · A single point moves the line: leverage versus residual

**Type**: Break the model

## Objective

Add the same vertical outlier in two different positions and measure how much it moves the slope: in the center of the
data almost nothing happens, at the edges the model breaks. Then compare robust estimators.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import HuberRegressor, LinearRegression, RANSACRegressor

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 50)
y = 2 + 1.5 * x + rng.normal(size=50)

## Commented solution

In [ ]:
def slope(x, y, model=None):
    model = model if model is not None else LinearRegression()
    model.fit(x.reshape(-1, 1), y)
    estimator = getattr(model, "estimator_", model)      # RANSAC hides the real model in estimator_
    return float(estimator.coef_[0])

slope_base = slope(x, y)
print("slope on the clean data:", round(slope_base, 3))

In [ ]:
x_lev, y_lev = np.append(x, 10.0), np.append(y, -20.0)           # outlier at the edge of the range
x_center, y_center = np.append(x, 5.0), np.append(y, -20.0)      # same outlier, in the center

slope_lev = slope(x_lev, y_lev)
slope_center = slope(x_center, y_center)
delta_lev = slope_lev - slope_base
delta_center = slope_center - slope_base
print(f"slope: base {slope_base:.3f} | point at the edge {slope_lev:.3f} | point at the center {slope_center:.3f}")
print(f"change: edge {delta_lev:+.3f} | center {delta_center:+.3f}")

In [ ]:
def leverages(x):
    A = np.column_stack([np.ones_like(x), x])
    return np.diag(A @ np.linalg.inv(A.T @ A) @ A.T)

lev_A = leverages(x_lev)[-1]
lev_B = leverages(x_center)[-1]
print(f"leverage of the added point: at the edge {lev_A:.3f} | in the center {lev_B:.3f} | expected mean leverage {2 / 51:.3f}")

In [ ]:
slope_huber = slope(x_lev, y_lev, HuberRegressor())
slope_ransac = slope(x_lev, y_lev, RANSACRegressor(random_state=0))
print(f"with the point at the edge -> OLS {slope_lev:.3f} | Huber {slope_huber:.3f} | RANSAC {slope_ransac:.3f} | true 1.5")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(x, y, s=15, alpha=0.6)
ax.scatter([10], [-20], color="#9C4530", label="outlier")
grid = np.linspace(0, 10, 50)
for name, m in [("OLS", LinearRegression()), ("Huber", HuberRegressor())]:
    m.fit(x_lev.reshape(-1, 1), y_lev)
    ax.plot(grid, m.predict(grid.reshape(-1, 1)), label=name)
ax.legend(); ax.set(xlabel="x", ylabel="y");

## Check

In [ ]:
with verify():
    check(close(slope_base, np.polyfit(x, y, 1)[0]), "correct slope on the clean data",
          "slope_base should match the least squares slope")
    check(len(x_lev) == 51 and len(x_center) == 51, "you added a single point to each dataset",
          "x_lev and x_center must have 51 elements")
    check(abs(delta_lev) > 5 * abs(delta_center),
          "the same outlier moves the line MUCH more at the edges than in the center",
          "delta_lev should be at least 5 times delta_center: did you add y = -20 at x = 10 and at x = 5?")
    check(lev_A > 2 * lev_B, "the leverage of the point at the edge is much larger",
          "lev_A should be more than double lev_B")
    check(abs(sum(leverages(x)) - 2) < 1e-8, "the sum of the leverages is p = 2 (intercept + slope)",
          "the sum of the diagonal elements of the hat matrix must equal the number of parameters")
    check(abs(slope_huber - slope_base) < abs(delta_lev) / 3, "Huber recovers most of the true slope",
          "the Huber slope should stay much closer to slope_base than the OLS one")
    check(abs(slope_ransac - slope_base) < 0.1, "RANSAC ignores the outlier completely",
          "RANSAC should estimate a slope very close to the clean one")
done()

## What you learned

- The same vertical outlier (y = -20) moves the slope by about 0.39 if it sits at the edge and by 0.02 if it sits in the center:
  a factor of 20. In the center the point pulls the line down but up and down in a balanced way; at the edge it has a
  "leverage" and rotates the whole line.
- The **leverage** depends only on x (is the point far from the center?), the **residual** depends on y (is the point far
  from the line?). Influence requires both, and that is what Cook's distance combines.
- Huber replaces the square with a linear penalty for large residuals: it reduces the damage but does not eliminate it.
  RANSAC takes another philosophy: it looks for a subset of consistent points and ignores the others completely.

## Common pitfall

Looking for outliers by looking only at large residuals. A high-leverage point "pulls" the line toward itself and can have
a modest residual: it is the worst case, because the residual plot looks fine. You need leverage and Cook's
distance, not just the residuals.

## Interview question

<details><summary>Why is an outlier at the margins of the data range more dangerous than one in the center, at the same distance from the line?</summary>

Because it has high leverage: in the hat matrix the weight of an observation on its own prediction grows with the distance of
x from the center. A point in the center can be compensated by all the others, a point at the edge cannot: the line bends
toward it because few other points constrain it in that area. That is why you look at Cook's distance, which
combines residual and leverage.

</details>